# 05 What-if: discount cap
Scenario built on historical relationships, not a forecast. Method: repriced lines above the cap at the cap, with cost held fixed. Retention and cost-avoided are assumptions, tested across a range.

In [11]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
import pandas as pd, numpy as np
df = pd.read_csv("Data/Clean/saas_clean.csv")
df.columns = [c.replace(" ", "_") for c in df.columns]
df['List_Value'] = df['Sales'] / (1 - df['Discount'])
df['Cost'] = df['Sales'] - df['Profit']
df['Cost_to_List'] = df['Cost'] / df['List_Value']
df['Band'] = np.where(df['Discount'] > 0.20, 'Above 20%', '20% or less')
print(df[['Sales','Profit','Cost','List_Value']].sum().round(0))

Sales         2297201.0
Profit         286397.0
Cost          2010804.0
List_Value    2863935.0
dtype: float64


In [13]:
print(df.pivot_table(index='Product', columns='Band', values='Cost_to_List', aggfunc='mean').round(2))

Band                        20% or less  Above 20%
Product                                           
Alchemy                            0.56       0.53
Big Ol Database                    0.62       0.62
ChatBot Plugin                     0.68        NaN
ContactMatcher                     0.56       0.59
Data Smasher                       0.71        NaN
FinanceHub                         0.67       0.78
Marketing Suite                    0.81       0.80
Marketing Suite - Gold             0.83        NaN
OneView                            0.71       0.71
SaaS Connector Pack                0.67       0.68
SaaS Connector Pack - Gold         0.52        NaN
Site Analytics                     0.73       0.71
Storage                            0.64        NaN
Support                            0.56        NaN


In [14]:
def scenario(cap, retention, cost_avoided=1.0):
    above = df[df['Discount'] > cap]
    new_sales = above['List_Value'] * (1 - cap)
    new_profit = new_sales - above['Cost']
    after = retention * new_profit.sum() + (1 - retention) * (-(1 - cost_avoided) * above['Cost'].sum())
    before = above['Profit'].sum()
    return after - before

In [15]:
rows = []
for cap in [0.10, 0.20, 0.30]:
    n = int((df['Discount'] > cap).sum())
    for a in [1.0, 0.5, 0.0]:
        row = {'Cap': f"{int(cap*100)}%", 'Lines above cap': n, 'Cost avoided on lost orders': f"{int(a*100)}%"}
        for r in [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]:
            row[f"Keep {int(r*100)}%"] = round(scenario(cap, r, a)/1000, 1)
        rows.append(row)
res = pd.DataFrame(rows)
print("Profit change vs actual, in thousands")
res

Profit change vs actual, in thousands


,Cap,Lines above cap,Cost avoided on lost orders,Keep 100%,Keep 80%,Keep 60%,Keep 40%,Keep 20%,Keep 0%
0,10%,5102,100%,389.1,320.0,250.9,181.8,112.7,43.6
1,10%,5102,50%,389.1,200.2,11.2,-177.7,-366.7,-555.7
2,10%,5102,0%,389.1,80.3,-228.5,-537.3,-846.1,-1154.9
3,20%,1393,100%,219.2,202.4,185.7,168.9,152.1,135.4
4,20%,1393,50%,219.2,152.6,86.0,19.5,-47.1,-113.7
5,20%,1393,0%,219.2,102.8,-13.6,-130.0,-246.4,-362.8
6,30%,1166,100%,146.4,142.2,137.9,133.6,129.3,125.0
7,30%,1166,50%,146.4,103.7,61.0,18.2,-24.5,-67.3
8,30%,1166,0%,146.4,65.2,-15.9,-97.1,-178.3,-259.5


In [16]:
for cap in [0.10, 0.20, 0.30]:
    above = df[df['Discount'] > cap]
    new_sales = (above['List_Value'] * (1 - cap)).sum()
    print(f"Cap {int(cap*100)}%: break-even share of orders kept (no cost avoided) = {above['Sales'].sum()/new_sales:.0%}")

Cap 10%: break-even share of orders kept (no cost avoided) = 75%
Cap 20%: break-even share of orders kept (no cost avoided) = 62%
Cap 30%: break-even share of orders kept (no cost avoided) = 64%


In [17]:
os.makedirs("Excels", exist_ok=True)
res.to_excel("Excels/whatif_cap_scenarios.xlsx", index=False)

In [18]:
for d in [0.10, 0.20, 0.30]:
    df[f'Loss_at_{int(d*100)}'] = (df['Cost_to_List'] > (1 - d)).astype(int)
print(df.groupby('Product')[['Loss_at_10','Loss_at_20','Loss_at_30']].mean().round(2).sort_values('Loss_at_20', ascending=False))
print(df[['Loss_at_10','Loss_at_20','Loss_at_30']].mean().round(3))

                            Loss_at_10  Loss_at_20  Loss_at_30
Product                                                       
Marketing Suite                   0.11        0.48        0.96
Marketing Suite - Gold            0.37        0.48        0.99
Data Smasher                      0.07        0.30        0.42
FinanceHub                        0.06        0.19        0.61
Storage                           0.18        0.18        0.18
SaaS Connector Pack               0.04        0.13        0.34
ContactMatcher                    0.01        0.08        0.17
Site Analytics                    0.08        0.08        0.87
Support                           0.04        0.04        0.11
Alchemy                           0.00        0.00        0.07
Big Ol Database                   0.00        0.00        0.38
ChatBot Plugin                    0.00        0.00        0.63
OneView                           0.00        0.00        0.76
SaaS Connector Pack - Gold        0.00        0.00     

In [19]:
caps = {'Marketing Suite': 0.10, 'Marketing Suite - Gold': 0.10, 'Data Smasher': 0.10}
df['Cap'] = df['Product'].map(caps).fillna(0.20)
aff = df[df['Discount'] > df['Cap']]
new_sales = aff['List_Value'] * (1 - aff['Cap'])
new_profit = new_sales - aff['Cost']

print("Lines affected:", len(aff))
print(aff.groupby('Product').size().sort_values(ascending=False))
print("Break-even share kept (no cost avoided):", f"{aff['Sales'].sum()/new_sales.sum():.0%}")

rows = []
for a in [1.0, 0.5, 0.0]:
    row = {'Cost avoided on lost orders': f"{int(a*100)}%"}
    for r in [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]:
        after = r*new_profit.sum() + (1-r)*(-(1-a)*aff['Cost'].sum())
        row[f"Keep {int(r*100)}%"] = round((after - aff['Profit'].sum())/1000, 1)
    rows.append(row)
pd.DataFrame(rows)

Lines affected: 2111
Product
ContactMatcher            789
Marketing Suite - Gold    316
Data Smasher              304
Marketing Suite           168
FinanceHub                158
SaaS Connector Pack       138
Site Analytics            109
OneView                    67
Big Ol Database            53
Alchemy                     9
dtype: int64
Break-even share kept (no cost avoided): 69%


,Cost avoided on lost orders,Keep 100%,Keep 80%,Keep 60%,Keep 40%,Keep 20%,Keep 0%
0,100%,243.6,221.1,198.7,176.3,153.9,131.4
1,50%,243.6,154.8,66.0,-22.8,-111.6,-200.4
2,0%,243.6,88.4,-66.7,-221.9,-377.0,-532.2


## Findings
- Scenario built on historical relationships, not a forecast. Cost held fixed per order line.
- 20% cap: +219k at full retention. Break-even: keep 62% of affected orders if lost orders save no cost.
- 10% cap: +389k at full retention, but break-even is 75% and it reprices 5,102 lines (51%), including profitable 20% discount lines.
- 30% cap: +146k, break-even 64%. Dominated by the 20% cap.
- Limits: uniform retention assumed; deeply discounted customers are likely to leave more. Marketing Suite and Marketing Suite - Gold have costs above 80% of list price, so they lose money even at a 20% discount.
- Product-specific view: Marketing Suite and Marketing Suite - Gold lose money on 48% of lines at a 20% discount; Gold still loses on 37% at 10%, so it needs a pricing or cost review, not just a cap. A tiered cap (10% on Marketing Suite, Marketing Suite - Gold and Data Smasher, 20% elsewhere) reprices 2,111 lines instead of 1,393. It adds about 24k at full retention but raises break-even from 62% to 69%, so it only beats the flat 20% cap if more than about 87% of affected orders are kept. Recommendation: flat 20% cap first, tiered caps only with evidence on price sensitivity.